# 03. 한국어 저랭크(LoRA) 추가학습 + 평가 (A100 40GB, 약 2시간 / 약 10~12유닛)
`ko-k32`(전문가 32개)에 한국어 지시문(KULLM-v2, 칼질 보정에 안 쓴 6천 개)으로 LoRA를 학습합니다.
- 어텐션 + 공유 전문가 + 남은 전문가에 LoRA, 라우터는 고정
- 학습 후 합쳐서 GGUF(Q6_K, Q4_K_M)로 만들고 **이 노트북에서 바로 평가**합니다
- 속도·전력은 같은 크기의 `ko-k32` 값을 그대로 씁니다 (구조와 크기가 같음)

In [ ]:
# 저장소 받기 + 패키지 설치
BRANCH = "ccr-a463bea6-bpbyrh"   # main에 합친 뒤에는 "main"
REPO = "github.com/b1bibig/Cfinetune"
import os
try:  # 저장소가 비공개면 코랩 왼쪽 열쇠(보안 비밀)에 GH_TOKEN 을 넣어 두세요
    from google.colab import userdata
    _tok = userdata.get("GH_TOKEN")
    REPO_URL = f"https://{_tok}@{REPO}"
except Exception:
    REPO_URL = f"https://{REPO}"
if not os.path.exists("/content/Cfinetune"):
    !git clone -q -b {BRANCH} {REPO_URL} /content/Cfinetune
%cd /content/Cfinetune
!git pull -q
%pip install -q -r requirements-colab.txt

In [ ]:
# 구글 드라이브: 결과(results/)를 세션이 끊겨도 남도록 저장
from google.colab import drive
drive.mount("/content/drive")
import os, shutil, sys
DRIVE = "/content/drive/MyDrive/cfinetune"
os.makedirs(DRIVE, exist_ok=True)

def pull_results():
    if os.path.exists(f"{DRIVE}/results"):
        shutil.copytree(f"{DRIVE}/results", "results", dirs_exist_ok=True)

def push_results():
    shutil.copytree("results", f"{DRIVE}/results", dirs_exist_ok=True)
    print("드라이브에 결과 저장함")

pull_results()
sys.path.insert(0, "scripts"); sys.path.insert(0, "src")

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU 메모리 {mem:.0f}GB")
assert mem >= 38, "런타임 > 런타임 유형 변경에서 A100 를 고르세요"

In [ ]:
!bash scripts/build_llama_cpp.sh /content/llama.cpp {DRIVE}/cache

In [ ]:
from _common import resolve_model_dir, load_config
cfg = load_config()
base = cfg["lora"]["base_variant"]
SRC = resolve_model_dir(cfg["model"]["id"])
!python scripts/02_prune.py --variant {base} --src {SRC} --out-dir /content/models/pruned
shutil.rmtree(SRC)   # 디스크 확보 (원본은 더 이상 필요 없음)

In [ ]:
OUT = f"/content/lora/{base}-lora"
!python scripts/04_train_lora.py --model /content/models/pruned/{base} --out {OUT}
shutil.copytree(f"{OUT}/adapter", f"{DRIVE}/lora/{base}-lora/adapter", dirs_exist_ok=True)
push_results()

In [ ]:
QS = " ".join(cfg["lora"]["quant_after"])
!python scripts/03_make_gguf.py --hf-dir {OUT}/merged --name {base}-lora --out-dir /content/gguf --quant {QS}
for q in cfg["lora"]["quant_after"]:
    g = f"/content/gguf/{base}-lora-{q}.gguf"
    !python scripts/05_eval.py --gguf {g} --name {base}-lora-{q}
    push_results()